# SemEval-2020 Task 7: Assessing Humor in Edited News Headlines | Subtask 1: Estimate the funniness of an edited headline on a 0-3 humor scale

**Team 1:** Matthew Vu, Vu Doan, Prapurna Penmathsa, and Amrisha Dwivedi

**Description:** The code in this Jupyter Notebook file performs all the necessary operations to accomplish SemEval-2020 Task 7: Assessing Humor in Edited News Headlines | Subtask 1: Estimate the funniness of an edited headline on a 0-3 humor scale.

**Acknowledgements:** This code is heavily based on the work on the work of Nabil Hossain (https://github.com/n-hossain/semeval-2020-task-7-humicroedit/tree/master) and Salih Tuncer (https://github.com/SalihTuncer/AssessHumor).

## Step 0: Import libraries and Set Seeds

In [24]:
import pandas as pd
import numpy as np
import random
import sys
import os
import tensorflow as tf
from transformers import AlbertTokenizer
from tensorflow.keras.optimizers.legacy import RMSprop
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from transformers import TFAlbertForSequenceClassification

In [25]:
np.random.seed(0)
random.seed(0)
tf.random.set_seed(0)

## Step 1: Baseline RMSE values

Recall, we have a validation set called "dev.csv" that is used to develop models, tune hyperparameters, and choose the optimal model out of a series of possible models.  Also, we have a test set called "test.csv" that is used to evaluate the final, best model to get an estimate on how well the chosen model will perform on unseen data in the future.

So, the first step in our analysis is to compute two baseline root mean square error (RMSE) values.  One baseline RMSE value is equal to the validation set's ("dev.csv") RMSE when the predicted funniness score of all edited news headlines in that validation set ("dev.csv") is equal to the average mean funniness score of all edited news headlines in the training set ("train.csv").  

The other baseline RMSE value is equal to the test set's ("test.csv") RMSE when the predicted funniness score of all edited news headlines in the test set ("test.csv") is equal to the average mean funniness score of all edited news headlines in the training set ("train.csv").

We want our chosen model to perform better than both the validation and test set baseline RMSE values.

In [26]:
# this function reads in the training set and the test/validation set 
# and writes an output file that is a copy of the test/validation set
# with the baseline predictions
# inputs:
# train_loc -> directory of the training set
# test_loc -> directory of the test/validation set
# output_name -> name of the desired output file
def baseline_dataset(train_loc, test_loc, output_name):
    # read in the training and test sets
    train = pd.read_csv(train_loc)    
    test = pd.read_csv(test_loc)
    
    # make predictions = average of meanGrade
    pred = np.mean(train['meanGrade'])    
    test['pred'] = pred
    
    # create and write to output file
    output = test[['id','pred']] 
    out_loc = f'datasets/subtask1/baseline/{output_name}'
    output.to_csv(out_loc, index=False)
    
    # show that the output is written
    print(f'{output_name} successfully written.')

In [27]:
# this function reads in the test/validation set and a dataset containing the baseline 
# predictions to compute the baseline RMSE of that test/validation set
# and returns the baseline RMSE
def baseline_RMSE(truth_loc, prediction_loc):
    # read in the true values
    truth = pd.read_csv(truth_loc, usecols=['id','meanGrade'])
    # read in the baseline predictions
    pred = pd.read_csv(prediction_loc, usecols=['id','pred'])
    
    assert(sorted(truth.id) == sorted(pred.id)),"ID mismatch between ground truth and prediction!"
    
    # compute the baseline RMSE
    data = pd.merge(truth,pred)
    rmse = np.sqrt(np.mean((data['meanGrade'] - data['pred'])**2))
    
    # return baseline RMSE
    return rmse

In [28]:
# create validation set baseline dataset
baseline_dataset('datasets/subtask1/train.csv', 'datasets/subtask1/dev.csv', 'dev_baseline.csv')

dev_baseline.csv successfully written.


In [29]:
# compute the output the validation set RMSE value
validation_baseline = baseline_RMSE('datasets/subtask1/dev.csv', 'datasets/subtask1/baseline/dev_baseline.csv')
print(f'Baseline validation set RMSE = {validation_baseline:.3f}')

Baseline validation set RMSE = 0.578


In [30]:
# create test set baseline dataset
baseline_dataset('datasets/subtask1/train.csv', 'datasets/subtask1/test.csv', 'test_baseline.csv')

test_baseline.csv successfully written.


In [31]:
# compute the output the test set RMSE value
test_baseline = baseline_RMSE('datasets/subtask1/test.csv', 'datasets/subtask1/baseline/test_baseline.csv')
print(f'Baseline test set RMSE = {test_baseline:.3f}')

Baseline test set RMSE = 0.575


We want our final model to get a better baseline validation and test set RMSE of 0.578 and 0.575, respectively.

## Step 2: Data Processing Code

In [32]:
# declare a class called Preprocess that contains all the methods to process the datasets
class Preprocess:

    # initialize the Class
    # inputs:
    # train_path -> the directory of the training data
    # dev_path -> the directory of the validation data
    # test.csv -> the directory of the test data
    # max_seq_length -> ?
    # batch_size -> ?
    def __init__(self,
                 train_path='datasets/subtask1/train.csv',
                 dev_path='datasets/subtask1/dev.csv',
                 test_path='datasets/subtask1/test.csv',
                 max_seq_length=256, batch_size=16
                 ):
        self._max_seq_length = max_seq_length

        # initialize the pre-trained Albert tokenizer
        self._tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')
        print('Tokenizer ready...\n')

        # read in the training, validation, and test sets
        train_df = pd.read_csv(train_path)
        dev_df = pd.read_csv(dev_path)
        test_df = pd.read_csv(test_path)
        print('Datasets were read...\n')

        # save the data appropriate as a class property
        # process the training set
        self._train = self._process_corpus(train_df)
        self._train = tf.data.Dataset.from_tensor_slices(self._train).batch(batch_size)
        print('Training-dataset preprocessed and ready...\n')

        # process the validation set
        self._dev = self._process_corpus(dev_df)
        self._dev = tf.data.Dataset.from_tensor_slices(self._dev).batch(batch_size)
        print('Dev-dataset preprocessed and ready...\n')

        # process test set 
        self._test = self._process_corpus(test_df)
        self._test = tf.data.Dataset.from_tensor_slices(self._test).batch(batch_size)
        print('Test-dataset preprocessed and ready...\n')

        print('Preprocessing done.\n')

    # process corpus
    def _process_corpus(self, df: pd.DataFrame) -> ({str: np.ndarray}, np.ndarray):
        # create a 2-D array of zeros with shape (len(df), self._max_seq_length)
        # it will store the tokenized version of each sentence in the dataset
        tokens = np.zeros((len(df), self._max_seq_length), dtype='int32')

        # create a 2-D array of zeros with shape (len(df), self._max_seq_length)
        # it stores the attention mask for each sentence
        attention_mask = np.zeros((len(df), self._max_seq_length), dtype='int32')

        # extract the labels (meanGrade) that the model with be trained to predict
        labels = df['meanGrade'].to_numpy(dtype='float32')

        for i in range(len(df)):
            # reformat the sentence and return additionally the replaced word as a whole sentence
            edited_sentence, sentence = self.reformat_sentence(df['original'][i], df['edit'][i])
            # save the encoded sentences in each row of the matrix
            tokens[i, :] = self.tokenize_sentence(edited_sentence, sentence)

            sentence_size = len(sentence.split(' ')) + 2  # CLS-token at the beginning and SEP-token at the end

            total_sentence_size = (sentence_size * 2) - 1  # second sentence has no CLS-token
            # fill the array with as many ones as we have words
            attention_mask[i, :total_sentence_size] = 1.0

        # return a dictionary containing 
        # inputs_id ->  2D numpy array containing the tokenized version of each sentence in the training dataset
        # attention_mask ->  the attention mask to indicate which tokens in each sentence should be attended to by the model
        # labels -> the target labels
        return {'input_ids': tokens,
                'attention_mask': attention_mask
                }, labels

    @staticmethod
    def reformat_sentence(sentence: str, edit: str) -> (str, str):
        # remove <..../> for the original sentence
        sen = sentence.split('<')
        sen[1] = sen[1].replace('/>', '')
        # and put the edited word in the <..../> and save it as a sentence
        sen[1:2] = sen[1].split(' ', 1)
        edited = sen[0] + edit + sen[2]
        return edited, ''.join(sen)

    # [CLS], ..., [SEP], ..., [SEP] | rest filled with [PAD]
    def tokenize_sentence(self, sentence: str, edited_sentence: str) -> [int]:
        # we want to place the special tokens ourself because we need 1 cls and 2 seps
        tokens = [self._tokenizer.cls_token_id] + self._tokenizer.encode(sentence, add_special_tokens=False) \
                 + [self._tokenizer.sep_token_id] + self._tokenizer.encode(edited_sentence, add_special_tokens=False) \
                 + [self._tokenizer.sep_token_id]
        # now we add the PAD-tokens at the end as filler tokens
        return tokens + [self._tokenizer.pad_token_id] * (self._max_seq_length - len(tokens))

    # get the training set
    def get_train(self) -> tf.data.Dataset:
        return self._train

    # get the validation set
    def get_dev(self) -> tf.data.Dataset:
        return self._dev

    # get the test set
    def get_test(self) -> tf.data.Dataset:
        return self._test

In [33]:
# declare the model class
class NN:
    # initialize the class
    def __init__(self):
        self._nn = self._create_nn()

    # returns an instance of TFAlbertforSequenceClassification
    def _create_nn(self) -> TFAlbertForSequenceClassification:
        # create a new instance of TFAlbertForSequenceClassification using a pre-trained
        # ALBERT model called 'albert-base-v2 to predict a regression task with num_labels = 1
        return TFAlbertForSequenceClassification.from_pretrained('albert-base-v2', num_labels=1)

    # returns the ALBERT model
    def get_nn(self):
        return self._nn

In [35]:
# preprocess the datasets and return a utility class which carries the datasets
max_seq_length = 256
batch_size = 64
prep = Preprocess(max_seq_length=max_seq_length, batch_size=batch_size)

# get the processed training and validation set
train_dataset = prep.get_train()
dev_dataset = prep.get_dev()

# create an albert model specialised on regression
model = NN().get_nn()

# use the RMSprop optimizer
optimizer = RMSprop(learning_rate = 1e-5, decay = 1e-8)

# compile the model with the appropriate optimizer, loss function, and evaluation metric
model.compile(optimizer=optimizer, loss=MeanSquaredError(),
            metrics=[RootMeanSquaredError('root_mean_squared_error')])

print("Fit model on training data.")

# train the model on the training set for 3 epochs
# and dusing the validation set for validation
model.fit(train_dataset, validation_data=dev_dataset, epochs=3)

print('Model successfully saved.')

# output the model summary including the layers in the model,
# the output shape of each layer, and the number of parameters in each layer 
print(model.summary())

# get the processed test set
test_dataset = prep.get_test()

print('Evaluate model with test data.')

# evaluate the test set
print(model.evaluate(test_dataset, return_dict=True))

print('Save the model weights.')

# save the model's weights
model.save_weights('rmsprop/rmsprop')

Tokenizer ready...

Datasets were read...

Training-dataset preprocessed and ready...

Dev-dataset preprocessed and ready...

Test-dataset preprocessed and ready...

Preprocessing done.



All PyTorch model weights were used when initializing TFAlbertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFAlbertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fit model on training data.
Epoch 1/3
 25/151 [===>..........................] - ETA: 3:05:14 - loss: 0.3895 - root_mean_squared_error: 0.6241

KeyboardInterrupt: 